In [ ]:
from pathlib import Path
from datetime import date, timedelta
import sys


BASE_DIR = Path.home() / "Documents" / "invesment-portfolio-analysis"
PROCESSED_DIR = BASE_DIR / "processed"
LAST_TRADE_DATE_FILE = PROCESSED_DIR / "last_trade_date.txt"


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


last_date = None
if LAST_TRADE_DATE_FILE.exists():
    last_line = next((ln.strip() for ln in LAST_TRADE_DATE_FILE.read_text(encoding="utf-8").splitlines() if ln.strip()), None)
    if last_line:
        try:
            last_date = date.fromisoformat(last_line)  # expects 'YYYY-MM-DD'
            print(f"Last trade date (from file): {last_date}")
        except ValueError:
            print(f"Invalid date '{last_line}' in {LAST_TRADE_DATE_FILE}; proceeding with full run.")
    else:
        print(f"No date found in {LAST_TRADE_DATE_FILE}; proceeding with full run.")
else:
    print(f"Missing {LAST_TRADE_DATE_FILE}; proceeding with full run.")


if last_date is not None:
    due_date = last_date + timedelta(days=10)
    today = date.today()
    if today < due_date:
        print(f"Not due yet. Last trade: {last_date}, due: {due_date}, today: {today}. Stopping.")
       
        raise SystemExit
    else:
        print(f"Due reached. Last trade: {last_date}, due: {due_date}, today: {today}. Continue notebook.")


Last trade date (from file): 2025-10-23
Not due yet. Last trade: 2025-10-23, due: 2025-11-02, today: 2025-10-24. Stopping.


SystemExit: 

In [371]:

import os
import sys
import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime
from bs4 import BeautifulSoup
import yfinance as yf
from dotenv import load_dotenv
from openai import OpenAI
import requests
from IPython.display import Markdown


In [354]:
from pathlib import Path
import subprocess, sys  # only needed if you call the scripts

# Base and folders
BASE_DIR = Path.home() / "Documents" / "invesment-portfolio-analysis"
FRONTEND_DIR = BASE_DIR / "frontend"
BACKEND_DIR  = BASE_DIR / "backend"
PROCESSED_DIR = BASE_DIR / "processed"
MODELS_DIR = BACKEND_DIR / "models" / "artifacts"

# Ensure directories exist
for d in [FRONTEND_DIR, BACKEND_DIR, PROCESSED_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Existing files
MODEL_5D_FILE = PROCESSED_DIR / "model.joblib"
MODEL_10D_FILE = MODELS_DIR / "model_lightgbm_optuna.joblib"
FEATURE_FILE = MODELS_DIR / "feature_columns.csv"
FEATURES_5D_JSON = PROCESSED_DIR / "feature_cols.json"
TOP10_JSON = PROCESSED_DIR / "top10_uptrend.json"
LAST_TRADE_DATE_FILE = PROCESSED_DIR / "last_trade_date.txt"

# NEW: portable script paths
backend_script  = BACKEND_DIR / "risk.py"
frontend_script= FRONTEND_DIR / "risk_analyzer.py"


# Optional: existence report
for f in [MODEL_5D_FILE, MODEL_10D_FILE, FEATURE_FILE, FEATURES_5D_JSON, TOP10_JSON, LAST_TRADE_DATE_FILE, BACKEND_SCRIPT, FRONTEND_SCRIPT]:
    print(f"{f} -> {'Exists ' if f.exists() else 'Missing ⚠'}")



/Users/ishanlahiru/Documents/invesment-portfolio-analysis/processed/model.joblib -> Exists 
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/backend/models/artifacts/model_lightgbm_optuna.joblib -> Exists 
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/backend/models/artifacts/feature_columns.csv -> Exists 
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/processed/feature_cols.json -> Exists 
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/processed/top10_uptrend.json -> Exists 
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/processed/last_trade_date.txt -> Exists 
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/backend/risk.py -> Exists 
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/frontend/risk_analyzer.py -> Exists 


In [332]:



load_dotenv(os.path.join(BASE_DIR, ".env"))
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = None
try:
    client = OpenAI(api_key=OPENAI_API_KEY)
except:
    print("⚠ OpenAI client not initialized")

In [333]:

# Load top 10 uptrend symbols

if os.path.exists(TOP10_JSON):
    with open(TOP10_JSON, "r") as f:
        top10_symbols = json.load(f)
else:
    top10_symbols = ["INTC","NKE","AAPL","TSM","CRM","TSLA","SAP","ORCL","ABBV","SONY"]

print("Top 10 symbols:", top10_symbols)

Top 10 symbols: ['INTC', 'NKE', 'AAPL', 'TSM', 'CRM', 'TSLA', 'SAP', 'ORCL', 'ABBV', 'SONY']


In [334]:

# Load models and features

model_5d = joblib.load(MODEL_5D_FILE)
model_10d = joblib.load(MODEL_10D_FILE)
with open(FEATURES_5D_JSON, "r") as f:
    feature_cols_5d = json.load(f)

feature_cols_10d = [
    "ema_12","momentum_10","atr_14","volume_sma_5","log_return_lag_1",
    "month_cos","tr","price_std_10","price_std_5","macd_signal","marketCap",
    "turnover_mktcap","rsi_14","turnover","log_return_20d","volatility_10d",
    "price_over_sma20","bb_percent_b_20","y_10d_log_return","volatility_5d",
    "drawdown_60","y_10d_price"
]

In [335]:
history_days = 200
data_list = []

for symbol in top10_symbols:
    ticker = yf.Ticker(symbol)
    hist = ticker.history(period=f"{history_days}d", interval="1d").reset_index()
    if hist.empty:
        print(f"⚠ No data for {symbol}")
        continue

    hist["symbol"] = symbol
    hist["name"] = symbol
    info = ticker.info
    hist["marketCap"] = info.get("marketCap", pd.NA)
    hist["turnover"] = hist["Close"] * hist["Volume"]
    hist["previousClose"] = hist["Close"].shift(1)  
    hist["change"] = hist["Close"] - hist["previousClose"]
    hist["percentageChange"] = (hist["change"] / hist["previousClose"]) * 100
    hist["tradevolume"] = hist["Volume"] 
    hist["marketCapPercentage"] = pd.NA  
    hist["closingPrice"] = hist["Close"]
    hist["crossingVolume"] = pd.NA  
    hist["crossingTradeVol"] = pd.NA 
    hist["status"] = pd.NA  
    hist["quantity"] = pd.NA 
    hist["id"] = pd.NA  
    data_list.append(hist)
    print(f" Fetched {len(hist)} rows for {symbol}")


master_df = pd.concat(data_list, ignore_index=True)
master_df.rename(columns={"Date":"lastTradedTime","Open":"open","High":"high","Low":"low",
                          "Volume":"sharevolume","Close":"price"}, inplace=True)
master_df['lastTradedTime'] = pd.to_datetime(master_df['lastTradedTime'], errors='coerce')


columns_order = [
    "id","name","symbol","quantity","percentageChange","change","price","previousClose",
    "high","low","turnover","sharevolume","tradevolume","marketCap","marketCapPercentage",
    "open","closingPrice","crossingVolume","crossingTradeVol","status","lastTradedTime"
]

master_df = master_df[columns_order]
master_df.head(10)

 Fetched 200 rows for INTC
 Fetched 200 rows for NKE
 Fetched 200 rows for AAPL
 Fetched 200 rows for TSM
 Fetched 200 rows for CRM
 Fetched 200 rows for TSLA
 Fetched 200 rows for SAP
 Fetched 200 rows for ORCL
 Fetched 200 rows for ABBV
 Fetched 200 rows for SONY


,id,name,symbol,quantity,percentageChange,change,price,previousClose,high,low,...,sharevolume,tradevolume,marketCap,marketCapPercentage,open,closingPrice,crossingVolume,crossingTradeVol,status,lastTradedTime
0,NaN,INTC,INTC,NaN,NaN,NaN,20.010000,NaN,20.340000,19.900000,...,61726100,61726100,181535424512,NaN,20.020000,20.010000,NaN,NaN,NaN,2025-01-07 00:00:00-05:00
1,NaN,INTC,INTC,NaN,-0.649680,-0.130001,19.879999,20.010000,20.120001,19.709999,...,47897100,47897100,181535424512,NaN,19.830000,19.879999,NaN,NaN,NaN,2025-01-08 00:00:00-05:00
2,NaN,INTC,INTC,NaN,-3.672030,-0.730000,19.150000,19.879999,19.660000,18.910000,...,71244600,71244600,181535424512,NaN,19.660000,19.150000,NaN,NaN,NaN,2025-01-10 00:00:00-05:00
3,NaN,INTC,INTC,NaN,0.261103,0.050001,19.200001,19.150000,19.250000,18.730000,...,50669000,50669000,181535424512,NaN,18.920000,19.200001,NaN,NaN,NaN,2025-01-13 00:00:00-05:00
4,NaN,INTC,INTC,NaN,0.000000,0.000000,19.200001,19.200001,19.520000,18.820000,...,47147500,47147500,181535424512,NaN,19.360001,19.200001,NaN,NaN,NaN,2025-01-14 00:00:00-05:00
5,NaN,INTC,INTC,NaN,2.708326,0.519999,19.719999,19.200001,19.770000,19.170000,...,56306000,56306000,181535424512,NaN,19.400000,19.719999,NaN,NaN,NaN,2025-01-15 00:00:00-05:00
6,NaN,INTC,INTC,NaN,-0.253546,-0.049999,19.670000,19.719999,19.950001,19.400000,...,44898100,44898100,181535424512,NaN,19.750000,19.670000,NaN,NaN,NaN,2025-01-16 00:00:00-05:00
7,NaN,INTC,INTC,NaN,9.252667,1.820000,21.490000,19.670000,21.620001,20.790001,...,166483900,166483900,181535424512,NaN,21.260000,21.490000,NaN,NaN,NaN,2025-01-17 00:00:00-05:00
8,NaN,INTC,INTC,NaN,1.302935,0.280001,21.770000,21.490000,22.410000,21.750000,...,96262200,96262200,181535424512,NaN,22.030001,21.770000,NaN,NaN,NaN,2025-01-21 00:00:00-05:00
9,NaN,INTC,INTC,NaN,0.413414,0.090000,21.860001,21.770000,22.290001,21.570000,...,60972600,60972600,181535424512,NaN,21.730000,21.860001,NaN,NaN,NaN,2025-01-22 00:00:00-05:00


In [336]:

df = master_df.copy()

df["log_return_5d"] = np.log(df["price"] / df["price"].shift(5))
df["volatility_5d"] = df["log_return_5d"].rolling(5).std()
df["price_std_5"] = df["price"].rolling(5).std()
df["y_5d_log_return"] = np.log(df["price"].shift(-5) / df["price"])
df["y_5d_price"] = df["price"].shift(-5)
df["log_return_lag_1"] = df["log_return_5d"].shift(1)


df["ema_12"] = df["price"].ewm(span=12, adjust=False).mean()
df["momentum_10"] = df["price"] - df["price"].shift(10)
df["atr_14"] = (df["high"] - df["low"]).rolling(14).mean()
df["volume_sma_5"] = df["sharevolume"].rolling(5).mean()
df["month_cos"] = np.cos(df["lastTradedTime"].dt.month * 2 * np.pi / 12)
df["price_std_10"] = df["price"].rolling(10).std()
df["macd_signal"] = df["price"].ewm(span=12).mean() - df["price"].ewm(span=26).mean()
df["turnover_mktcap"] = df["turnover"] / df["marketCap"]
df["rsi_14"] = (
    df["price"].diff().clip(lower=0).rolling(14).mean()
    / df["price"].diff().abs().rolling(14).mean()
) * 100
df["tr"] = df["high"] - df["low"]
df["bb_percent_b_20"] = (
    (df["price"] - df["price"].rolling(20).mean()) / (2 * df["price"].rolling(20).std())
)
df["price_over_sma20"] = df["price"] / df["price"].rolling(20).mean()
df["drawdown_60"] = df["price"] / df["price"].rolling(60).max() - 1


feature_cols = [
    "log_return_5d","volatility_5d","price_std_5","log_return_lag_1","ema_12",
    "momentum_10","atr_14","volume_sma_5","month_cos","price_std_10",
    "macd_signal","turnover_mktcap","rsi_14","tr","bb_percent_b_20",
    "price_over_sma20","drawdown_60"
]
df[feature_cols] = df[feature_cols].fillna(method="bfill").fillna(method="ffill")


required_cols = [
    "name","symbol","lastTradedTime","ema_12","momentum_10","atr_14",
    "volume_sma_5","log_return_lag_1","month_cos","tr","price_std_10",
    "price_std_5","macd_signal","marketCap","turnover_mktcap","rsi_14",
    "turnover","log_return_5d","volatility_5d","price_over_sma20",
    "bb_percent_b_20","y_5d_log_return","drawdown_60","y_5d_price"
]

df_5d = df[required_cols].copy()

# --- Drop rows where the target (future price) is NaN ---
df_5d = df_5d.dropna(subset=["y_5d_price"]).reset_index(drop=True)

print(f"5-Day Features prepared successfully: {df_5d.shape}")

df_5d = df_5d.dropna(subset=["y_5d_price"]).reset_index(drop=True)

print(f"5-Day Features prepared successfully: {df_5d.shape}")
df_5d.head(10)


5-Day Features prepared successfully: (1995, 24)
5-Day Features prepared successfully: (1995, 24)


/var/folders/d5/p58vb38175d8xw9s73_rl4g00000gn/T/ipykernel_57789/729423392.py:37: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[feature_cols] = df[feature_cols].fillna(method="bfill").fillna(method="ffill")


,name,symbol,lastTradedTime,ema_12,momentum_10,atr_14,volume_sma_5,log_return_lag_1,month_cos,tr,...,turnover_mktcap,rsi_14,turnover,log_return_5d,volatility_5d,price_over_sma20,bb_percent_b_20,y_5d_log_return,drawdown_60,y_5d_price
0,INTC,INTC,2025-01-07 00:00:00-05:00,20.010000,1.559999,0.625715,55736860.0,-0.014599,0.866025,0.440001,...,0.006804,47.750863,1.235139e+09,-0.014599,0.074778,0.977734,-0.247702,-0.014599,-0.181088,19.719999
1,INTC,INTC,2025-01-08 00:00:00-05:00,19.990000,1.559999,0.625715,55736860.0,-0.014599,0.866025,0.410002,...,0.005245,47.750863,9.521943e+08,-0.014599,0.074778,0.977734,-0.247702,-0.010620,-0.181088,19.670000
2,INTC,INTC,2025-01-10 00:00:00-05:00,19.860769,1.559999,0.625715,55736860.0,-0.014599,0.866025,0.750000,...,0.007516,47.750863,1.364334e+09,-0.014599,0.074778,0.977734,-0.247702,0.115285,-0.181088,21.490000
3,INTC,INTC,2025-01-13 00:00:00-05:00,19.759113,1.559999,0.625715,55736860.0,-0.014599,0.866025,0.520000,...,0.005359,47.750863,9.728448e+08,-0.014599,0.074778,0.977734,-0.247702,0.125623,-0.181088,21.770000
4,INTC,INTC,2025-01-14 00:00:00-05:00,19.673095,1.559999,0.625715,55736860.0,-0.014599,0.866025,0.700001,...,0.004987,47.750863,9.052320e+08,-0.014599,0.074778,0.977734,-0.247702,0.129748,-0.181088,21.860001
5,INTC,INTC,2025-01-15 00:00:00-05:00,19.680311,1.559999,0.625715,54652840.0,-0.014599,0.866025,0.600000,...,0.006116,47.750863,1.110354e+09,-0.014599,0.074778,0.977734,-0.247702,0.089670,-0.181088,21.570000
6,INTC,INTC,2025-01-16 00:00:00-05:00,19.678725,1.559999,0.625715,54053040.0,-0.014599,0.866025,0.550001,...,0.004865,47.750863,8.831456e+08,-0.010620,0.074778,0.977734,-0.247702,0.057300,-0.181088,20.830000
7,INTC,INTC,2025-01-17 00:00:00-05:00,19.957383,1.559999,0.625715,73100900.0,-0.010620,0.866025,0.830000,...,0.019708,47.750863,3.577739e+09,0.115285,0.074778,0.977734,-0.247702,-0.057460,-0.181088,20.290001
8,INTC,INTC,2025-01-21 00:00:00-05:00,20.236247,1.559999,0.625715,82219540.0,0.115285,0.866025,0.660000,...,0.011544,47.750863,2.095628e+09,0.125623,0.074778,0.977734,-0.247702,-0.094851,-0.181088,19.799999
9,INTC,INTC,2025-01-22 00:00:00-05:00,20.486055,1.559999,0.625715,84984560.0,0.125623,0.866025,0.720001,...,0.007342,47.750863,1.332861e+09,0.129748,0.074778,0.977734,-0.247702,-0.101505,-0.181088,19.750000


In [337]:

latest_5d = df_5d.groupby("symbol").tail(1).reset_index(drop=True)

def fill_missing_features(df, feature_cols):
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    df[feature_cols] = (
        df[feature_cols]
        .bfill()   
        .ffill()   
        .fillna(0)
    )
    return df

latest_5d = fill_missing_features(latest_5d, feature_cols_5d)


latest_prices = master_df.groupby("symbol")["price"].last().reset_index()
latest_5d = latest_5d.drop(columns=["price"], errors="ignore")
latest_5d = latest_5d.merge(latest_prices, on="symbol", how="left")


latest_5d = latest_5d.reset_index(drop=True)
latest_5d["pred_5d_log_return"] = model_5d.predict(latest_5d[feature_cols_5d])

latest_5d["pred_5d_price"] = latest_5d["price"] * np.exp(latest_5d["pred_5d_log_return"])

pred_5d_prices = latest_5d[["symbol", "pred_5d_price"]]
print("\nPredicted 5-day prices per symbol:")
print(pred_5d_prices.head(10))



Predicted 5-day prices per symbol:
  symbol  pred_5d_price
0   INTC      37.747685
1    NKE      69.181991
2   AAPL     263.388935
3    TSM     294.184763
4    CRM     252.365792
5   TSLA     453.098034
6    SAP     275.682107
7   ORCL     281.137648
8   ABBV     225.171358
9   SONY      28.486283


In [338]:
import pandas as pd
import numpy as np

# Copy master df
master_df_raw = master_df.copy()

# Convert data types 
master_df_raw['lastTradedTime'] = pd.to_datetime(master_df_raw['lastTradedTime'], errors='coerce')
num_cols = [
    'price','high','low','previousClose','turnover',
    'sharevolume','tradevolume','marketCap','change',
    'percentageChange','open','closingPrice'
]
for col in num_cols:
    if col in master_df_raw.columns:
        master_df_raw[col] = pd.to_numeric(master_df_raw[col], errors='coerce')

# Sort and group 
master_df_raw = master_df_raw.sort_values(['symbol','lastTradedTime']).reset_index(drop=True)
grouped = master_df_raw.groupby('symbol')

# Core returns & volatility 
master_df_raw['log_return_1d'] = grouped['price'].transform(lambda x: np.log(x / x.shift(1)))
for w in [5,10,20]:
    master_df_raw[f'log_return_{w}d'] = grouped['price'].transform(lambda x: np.log(x / x.shift(w)))
    master_df_raw[f'price_sma_{w}'] = grouped['price'].transform(lambda x: x.rolling(w).mean())
    master_df_raw[f'price_std_{w}'] = grouped['price'].transform(lambda x: x.rolling(w).std())
    master_df_raw[f'volatility_{w}d'] = grouped['log_return_1d'].transform(lambda x: x.rolling(w).std())

#  Momentum & ratios
master_df_raw['momentum_10'] = grouped['price'].transform(lambda x: x / x.shift(10) - 1)
master_df_raw['volatility_ratio'] = master_df_raw['price_std_10'] / (master_df_raw['price_std_20'] + 1e-9)
master_df_raw['price_over_sma20'] = master_df_raw['price'] / (master_df_raw['price_sma_20'] + 1e-9) - 1

#  Volume indicators 
master_df_raw['volume_sma_5'] = grouped['sharevolume'].transform(lambda x: x.rolling(5).mean())
master_df_raw['volume_ratio'] = master_df_raw['sharevolume'] / (master_df_raw['volume_sma_5'] + 1e-9)

#  MACD 
master_df_raw['ema_12'] = grouped['price'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
master_df_raw['ema_26'] = grouped['price'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
master_df_raw['macd'] = master_df_raw['ema_12'] - master_df_raw['ema_26']
master_df_raw['macd_signal'] = grouped['macd'].transform(lambda x: x.ewm(span=9, adjust=False).mean())
master_df_raw['macd_hist'] = master_df_raw['macd'] - master_df_raw['macd_signal']

#  Bollinger Bands 
for w in [20]:
    mean = grouped['price'].transform(lambda x: x.rolling(w).mean())
    std = grouped['price'].transform(lambda x: x.rolling(w).std())
    master_df_raw[f'bb_upper_{w}'] = mean + 2 * std
    master_df_raw[f'bb_lower_{w}'] = mean - 2 * std
    master_df_raw[f'bb_width_{w}'] = 4 * std
    master_df_raw[f'bb_percent_b_{w}'] = (master_df_raw['price'] - (mean - 2 * std)) / (4 * std + 1e-9)

# ATR & RSI 
master_df_raw['tr'] = np.maximum.reduce([
    master_df_raw['high'] - master_df_raw['low'],
    np.abs(master_df_raw['high'] - master_df_raw['previousClose']),
    np.abs(master_df_raw['low'] - master_df_raw['previousClose'])
])
master_df_raw['atr_14'] = grouped['tr'].transform(lambda x: x.rolling(14).mean())

def compute_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

master_df_raw['rsi_14'] = grouped['price'].transform(lambda x: compute_rsi(x))

#  Regime & liquidity 
master_df_raw['rolling_max_60'] = grouped['price'].transform(lambda x: x.rolling(60).max())
master_df_raw['drawdown_60'] = (master_df_raw['price'] - master_df_raw['rolling_max_60']) / (master_df_raw['rolling_max_60'] + 1e-9)
master_df_raw['turnover_mktcap'] = master_df_raw['turnover'] / (master_df_raw['marketCap'] + 1e-9)

#  Cyclic time features
master_df_raw['day_of_week'] = master_df_raw['lastTradedTime'].dt.dayofweek
master_df_raw['month'] = master_df_raw['lastTradedTime'].dt.month
master_df_raw['day_of_week_sin'] = np.sin(2 * np.pi * master_df_raw['day_of_week'] / 7)
master_df_raw['day_of_week_cos'] = np.cos(2 * np.pi * master_df_raw['day_of_week'] / 7)
master_df_raw['month_sin'] = np.sin(2 * np.pi * master_df_raw['month'] / 12)
master_df_raw['month_cos'] = np.cos(2 * np.pi * master_df_raw['month'] / 12)

# Cross-sectional z-scores 
for col in ['price', 'log_return_1d', 'volume_ratio']:
    if col in master_df_raw.columns:
        master_df_raw[f'{col}_zscore_xsec'] = master_df_raw.groupby('lastTradedTime')[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-9)
        )

# Lag features 
for lag in [1, 5]:
    master_df_raw[f'log_return_lag_{lag}'] = grouped['log_return_1d'].transform(lambda x: x.shift(lag))

# Clean missing values safely
core_cols = ['symbol', 'lastTradedTime', 'price']
master_df_raw = master_df_raw.dropna(subset=core_cols)

#  Forward/backward fill for remaining features safely 
cols_to_fill = [c for c in master_df_raw.columns if c not in core_cols]
for col in cols_to_fill:
    master_df_raw[col] = master_df_raw[col].infer_objects()  # convert object to proper dtype
    master_df_raw[col] = master_df_raw.groupby('symbol')[col].transform(lambda x: x.ffill().bfill())

#  Final features DataFrame 
features_df = master_df_raw.reset_index(drop=True)

print(" Features generated successfully for prediction!")
print("Shape:", features_df.shape)
display(features_df.tail(15))


 Features generated successfully for prediction!
Shape: (2000, 65)


,id,name,symbol,quantity,percentageChange,change,price,previousClose,high,low,...,month,day_of_week_sin,day_of_week_cos,month_sin,month_cos,price_zscore_xsec,log_return_1d_zscore_xsec,volume_ratio_zscore_xsec,log_return_lag_1,log_return_lag_5
1985,NaN,TSM,TSM,NaN,1.416132,4.080017,292.190002,288.109985,296.059998,290.489990,...,10,-0.433884,-0.900969,-0.866025,0.5,0.601003,0.958634,0.410422,-0.001249,-0.012000
1986,NaN,TSM,TSM,NaN,3.494299,10.209991,302.399994,292.190002,307.299988,299.940002,...,10,0.000000,1.000000,-0.866025,0.5,0.619573,0.857930,0.980820,0.014062,-0.000476
1987,NaN,TSM,TSM,NaN,-2.767856,-8.369995,294.029999,302.399994,307.100006,293.279999,...,10,0.781831,0.623490,-0.866025,0.5,0.609996,-0.682145,0.108128,0.034346,0.021937
1988,NaN,TSM,TSM,NaN,3.567660,10.489990,304.519989,294.029999,306.399994,293.980011,...,10,0.974928,-0.222521,-0.866025,0.5,0.660366,2.318364,1.399045,-0.028069,0.032340
1989,NaN,TSM,TSM,NaN,-1.523704,-4.639984,299.880005,304.519989,302.829987,297.420013,...,10,0.433884,-0.900969,-0.866025,0.5,0.627542,-0.995973,-0.454135,0.035055,-0.001249
1990,NaN,TSM,TSM,NaN,-6.409231,-19.220001,280.660004,299.880005,300.200012,280.329987,...,10,-0.433884,-0.900969,-0.866025,0.5,0.561712,-1.496929,1.404035,-0.015354,0.014062
1991,NaN,TSM,TSM,NaN,7.920620,22.230011,302.890015,280.660004,304.630005,292.320007,...,10,0.000000,1.000000,-0.866025,0.5,0.643236,1.971164,1.953808,-0.066238,0.034346
1992,NaN,TSM,TSM,NaN,-2.294566,-6.950012,295.940002,302.890015,301.750000,291.339996,...,10,0.781831,0.623490,-0.866025,0.5,0.627103,-0.478162,0.617058,0.076226,-0.028069
1993,NaN,TSM,TSM,NaN,2.963435,8.769989,304.709991,295.940002,306.609985,300.070007,...,10,0.974928,-0.222521,-0.866025,0.5,0.670161,1.160665,1.522681,-0.023213,0.035055
1994,NaN,TSM,TSM,NaN,-1.598239,-4.869995,299.839996,304.709991,311.369995,296.690002,...,10,0.433884,-0.900969,-0.866025,0.5,0.633703,-0.687660,0.182386,0.029204,-0.015354


In [339]:
#  Columns to keep for prediction 
predict_cols = [
    'id','name','symbol','lastTradedTime','ema_12','momentum_10','atr_14','volume_sma_5',
    'log_return_lag_1','month_cos','tr','price_std_10','price_std_5','macd_signal','marketCap',
    'turnover_mktcap','rsi_14','turnover','log_return_20d','volatility_10d','price_over_sma20',
    'bb_percent_b_20','volatility_5d','drawdown_60'
]

# Filter only these columns 
predict_df = features_df[predict_cols].copy()

#  Ensure numeric columns are numeric 
numeric_cols = predict_df.select_dtypes(include=['float64', 'int64']).columns
predict_df[numeric_cols] = predict_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

#  Forward/backward fill missing feature values per symbol (safer) 
feature_cols = [c for c in predict_cols if c not in ['id','name','symbol','lastTradedTime']]
for col in feature_cols:
    predict_df[col] = predict_df.groupby('symbol')[col].transform(lambda x: x.ffill().bfill())


print("Prediction dataframe ready!")
print("Shape:", predict_df.shape)
display(predict_df.tail(10))




Prediction dataframe ready!
Shape: (2000, 24)


,id,name,symbol,lastTradedTime,ema_12,momentum_10,atr_14,volume_sma_5,log_return_lag_1,month_cos,...,marketCap,turnover_mktcap,rsi_14,turnover,log_return_20d,volatility_10d,price_over_sma20,bb_percent_b_20,volatility_5d,drawdown_60
1990,NaN,TSM,TSM,2025-10-10 00:00:00-04:00,287.684563,0.026705,10.577853,15069020.0,-0.015354,0.5,...,1507877978112,0.004355,54.356083,6.566883e+09,0.079043,0.032450,0.000822,0.504268,0.043288,-0.078353
1991,NaN,TSM,TSM,2025-10-13 00:00:00-04:00,290.023863,0.108553,11.334996,15975680.0,-0.066238,0.5,...,1507877978112,0.004081,59.672176,6.154210e+09,0.147394,0.039853,0.072158,0.874429,0.055789,-0.005353
1992,NaN,TSM,TSM,2025-10-14 00:00:00-04:00,290.934038,0.059615,11.732141,16673060.0,0.076226,0.5,...,1507877978112,0.003252,56.968982,4.903104e+09,0.121583,0.040932,0.041313,0.725302,0.055211,-0.028175
1993,NaN,TSM,TSM,2025-10-15 00:00:00-04:00,293.053416,0.056297,11.742855,18266420.0,-0.023213,0.5,...,1507877978112,0.004201,62.303707,6.334311e+09,0.148005,0.040717,0.064322,0.860455,0.054372,0.000000
1994,NaN,TSM,TSM,2025-10-16 00:00:00-04:00,294.097505,0.040714,12.359996,21482740.0,0.029204,0.5,...,1507877978112,0.005238,61.457260,7.898235e+09,0.109877,0.041258,0.041636,0.741590,0.054427,-0.015982
1995,NaN,TSM,TSM,2025-10-17 00:00:00-04:00,294.248656,0.009891,12.484281,20329800.0,-0.016111,0.5,...,1507877978112,0.003451,59.089764,5.203234e+09,0.108007,0.041537,0.019748,0.627093,0.042473,-0.031604
1996,NaN,TSM,TSM,2025-10-20 00:00:00-04:00,294.779634,-0.015542,12.649283,18944460.0,-0.016003,0.5,...,1507877978112,0.002644,57.884368,3.986679e+09,0.087971,0.040016,0.024365,0.666080,0.021942,-0.023005
1997,NaN,TSM,TSM,2025-10-21 00:00:00-04:00,294.738153,0.001633,12.106426,17868980.0,0.008840,0.5,...,1507877978112,0.002186,52.726619,3.295714e+09,0.040891,0.039106,0.011336,0.578416,0.019736,-0.033474
1998,NaN,TSM,TSM,2025-10-22 00:00:00-04:00,293.836900,-0.051359,12.185713,16793200.0,-0.010773,0.5,...,1507877978112,0.002952,50.331819,4.451381e+09,0.028689,0.037461,-0.009387,0.433232,0.011327,-0.051951
1999,NaN,TSM,TSM,2025-10-23 00:00:00-04:00,293.358917,-0.030512,11.989997,13581433.2,-0.019302,0.5,...,1507877978112,0.001983,49.358528,2.989480e+09,0.049606,0.037441,-0.005443,0.458713,0.012973,-0.045880


In [340]:

model_10d = joblib.load(MODEL_10D_FILE)
X_cols_saved = pd.read_csv(FEATURE_FILE)["feature"].tolist()

# Fill numeric NaNs with 0
num_features = df_predict.select_dtypes(include=np.number).columns.tolist()
df_predict[num_features] = df_predict[num_features].fillna(0.0)

# Drop rows missing essential info
essential_cols = ["symbol", "lastTradedTime"]
df_predict = df_predict.dropna(subset=essential_cols).reset_index(drop=True)

# Ensure all saved features exist in df_predict
for col in X_cols_saved:
    if col not in df_predict.columns:
        df_predict[col] = 0.0

# Take the latest row per symbol for prediction
latest_predict = df_predict.groupby("symbol").tail(1).reset_index(drop=True)
X_latest = latest_predict[X_cols_saved]

# Predict 10-day price directly
latest_predict["pred_10d_price"] = model_10d.predict(X_latest)

# Fetch latest price from Yahoo Finance
symbols = latest_predict["symbol"].unique().tolist()
current_prices = {}

for sym in symbols:
    try:
        ticker = yf.Ticker(sym)
        hist = ticker.history(period="1d")
        if not hist.empty:
            current_prices[sym] = hist["Close"].iloc[-1]
        else:
            current_prices[sym] = 1.0  # fallback if no data
    except Exception as e:
        print(f"Could not fetch price for {sym}: {e}")
        current_prices[sym] = 1.0

latest_predict["current_price"] = latest_predict["symbol"].map(current_prices)

# Calculate log return for 10-day horizon
latest_predict["pred_10d_log_return"] = np.log(latest_predict["pred_10d_price"] / latest_predict["current_price"])

# Rank symbols by predicted log return
latest_predict = latest_predict.sort_values("pred_10d_log_return", ascending=False).reset_index(drop=True)
latest_predict["predicted_rank"] = latest_predict.index + 1

# Select final columns to display
result_cols = ["predicted_rank", "symbol", "lastTradedTime", "current_price",
               "pred_10d_log_return", "pred_10d_price"]

print("✅ Latest predicted 10-day stock prices:")
display(latest_predict[result_cols])


✅ Latest predicted 10-day stock prices:


,predicted_rank,symbol,lastTradedTime,current_price,pred_10d_log_return,pred_10d_price
0,1,TSM,2025-10-23 00:00:00-04:00,290.730011,0.062039,309.337770
1,2,ORCL,2025-10-23 00:00:00-04:00,280.070007,0.017887,285.124584
2,3,ABBV,2025-10-23 00:00:00-04:00,228.250000,0.016593,232.069054
3,4,SONY,2025-10-23 00:00:00-04:00,28.709999,-0.016568,28.238253
4,5,NKE,2025-10-23 00:00:00-04:00,69.680000,-0.019898,68.307244
5,6,CRM,2025-10-23 00:00:00-04:00,255.050003,-0.020496,249.875828
6,7,SAP,2025-10-23 00:00:00-04:00,278.529999,-0.034448,269.098502
7,8,AAPL,2025-10-22 00:00:00-04:00,259.579987,-0.063567,243.592685
8,9,TSLA,2025-10-23 00:00:00-04:00,448.980011,-0.089579,410.509526
9,10,INTC,2025-10-23 00:00:00-04:00,38.160000,-0.173068,32.095630


In [341]:
# ---------- Your existing merge and display ----------
combined_df = latest_predict.merge(
    latest_5d[["symbol", "pred_5d_price"]],
    on="symbol",
    how="left"
)

result_cols = [
    "predicted_rank",
    "symbol",
    "lastTradedTime",
    "current_price",
    "pred_5d_price",
    "pred_10d_price"
]

print(" Latest predicted stock prices with 5-day and 10-day predictions:")
display(combined_df[result_cols])

# ---------- Save most recent last trade DATE ----------
last_traded_ts = pd.to_datetime(combined_df["lastTradedTime"], errors="coerce").max()
if pd.isna(last_traded_ts):
    raise ValueError("No valid lastTradedTime values found to save.")
last_trade_date_str = last_traded_ts.date().isoformat()  # e.g. '2025-10-24'

LAST_TRADE_DATE_FILE.write_text(last_trade_date_str + "\n", encoding="utf-8")
print(f"Saved last trade date to: {LAST_TRADE_DATE_FILE}")

 Latest predicted stock prices with 5-day and 10-day predictions:


,predicted_rank,symbol,lastTradedTime,current_price,pred_5d_price,pred_10d_price
0,1,TSM,2025-10-23 00:00:00-04:00,290.730011,294.184763,309.337770
1,2,ORCL,2025-10-23 00:00:00-04:00,280.070007,281.137648,285.124584
2,3,ABBV,2025-10-23 00:00:00-04:00,228.250000,225.171358,232.069054
3,4,SONY,2025-10-23 00:00:00-04:00,28.709999,28.486283,28.238253
4,5,NKE,2025-10-23 00:00:00-04:00,69.680000,69.181991,68.307244
5,6,CRM,2025-10-23 00:00:00-04:00,255.050003,252.365792,249.875828
6,7,SAP,2025-10-23 00:00:00-04:00,278.529999,275.682107,269.098502
7,8,AAPL,2025-10-22 00:00:00-04:00,259.579987,263.388935,243.592685
8,9,TSLA,2025-10-23 00:00:00-04:00,448.980011,453.098034,410.509526
9,10,INTC,2025-10-23 00:00:00-04:00,38.160000,37.747685,32.095630


Saved last trade date to: /Users/ishanlahiru/Documents/invesment-portfolio-analysis/processed/last_trade_date.txt


In [342]:

news_cache = {}
def get_stock_news(symbol, count=5):
    if symbol in news_cache:
        return news_cache[symbol]
    try:
        url = f"https://finviz.com/quote.ashx?t={symbol}"
        headers = {"User-Agent":"Mozilla/5.0"}
        resp = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(resp.content, "html.parser")
        news_table = soup.find(id="news-table")
        if not news_table:
            news_cache[symbol] = ["No news found."]
            return news_cache[symbol]
        rows = news_table.find_all("tr")[:count]
        news_list = []
        for row in rows:
            a_tag = row.find("a")
            if a_tag:
                news_list.append(f"- {a_tag.text.strip()} ([link]({a_tag['href']}))")
        news_cache[symbol] = news_list
        return news_list
    except Exception as e:
        news_cache[symbol] = [f"No news (error: {e})"]
        return news_cache[symbol]


symbols = latest_predict["symbol"].tolist()

for sym in symbols:
    news_list = get_stock_news(sym, count=5)
    display(Markdown(f"### News for {sym}"))
    for news in news_list:
        display(Markdown(news))


### News for TSM

- Intel stock jumps as Q3 earnings beat expectations, AI drives chip demand ([link](https://finance.yahoo.com/news/intel-stock-jumps-as-q3-earnings-beat-expectations-ai-drives-chip-demand-201344592.html))

- The Best Stocks to Invest $1,000 In Right Now ([link](/news/202673/the-best-stocks-to-invest-1000-in-right-now))

- Intel Q3 earnings set to test stock's rally amid Nvidia, US government investments ([link](https://finance.yahoo.com/news/intel-q3-earnings-set-to-test-stocks-rally-amid-nvidia-us-government-investments-150922622.html))

- Intel Q3 earnings beat Wall Street estimates but Q4 outlook falls short ([link](https://finance.yahoo.com/news/intel-q3-earnings-beat-wall-street-estimates-but-q4-outlook-falls-short-150922457.html))

...

- Intel Stock Rises On Big Third-Quarter Earnings Beat ([link](https://finance.yahoo.com/m/1ed564d1-3d69-3125-9aec-13641bf3a966/intel-stock-rises-on-big.html))

- Intels Q3 results show signs of recovery  and restraint ([link](https://finance.yahoo.com/m/8da0a8e1-223a-39ca-82f1-b58a7a1d98ee/intel%E2%80%99s-q3-results-show-signs.html))

- Intel beats expectations as investments help in turnaround ([link](https://finance.yahoo.com/news/intel-beats-expectations-investments-help-204905895.html))

- Intel stock jumps as Q3 earnings beat expectations, AI drives chip demand ([link](https://finance.yahoo.com/news/intel-stock-jumps-as-q3-earnings-beat-expectations-ai-drives-chip-demand-201344592.html))

- Intel posts profit even as it struggles to regain market share ([link](https://finance.yahoo.com/news/intel-posts-profit-even-struggles-204048976.html))

In [360]:
# Verify presence (optional)
for p in [backend_script, frontend_script]:
    print(f"{p} -> {'Exists ✅' if p.exists() else 'Missing ⚠'}")

# Run backend first, then frontend; raises if a script fails
subprocess.run([sys.executable, str(backend_script)], check=True)
subprocess.run([sys.executable, str(frontend_script)], check=True)

/Users/ishanlahiru/Documents/invesment-portfolio-analysis/backend/risk.py -> Exists ✅
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/frontend/risk_analyzer.py -> Exists ✅


/Users/ishanlahiru/Documents/invesment-portfolio-analysis/backend/risk.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


Fetching ~1M data for: INTC, NKE, AAPL, TSM, CRM, TSLA, SAP, ORCL, ABBV, SONY
✅ INTC: Vol=56.48%  Beta=1.26  VaR(N)=4.57%  VaR(H)=3.72%  Sharpe=5.75  Sortino=7.43
✅ NKE: Vol=36.29%  Beta=1.71  VaR(N)=3.85%  VaR(H)=3.50%  Sharpe=-0.57  Sortino=-0.49
✅ AAPL: Vol=22.61%  Beta=1.25  VaR(N)=2.28%  VaR(H)=1.64%  Sharpe=0.68  Sortino=0.69
✅ TSM: Vol=47.49%  Beta=2.68  VaR(N)=4.61%  VaR(H)=2.72%  Sharpe=1.67  Sortino=2.15
✅ CRM: Vol=37.46%  Beta=0.97  VaR(N)=3.77%  VaR(H)=3.23%  Sharpe=0.75  Sortino=0.82
✅ TSLA: Vol=50.56%  Beta=2.71  VaR(N)=5.06%  VaR(H)=5.00%  Sharpe=0.93  Sortino=0.96
✅ SAP: Vol=21.29%  Beta=1.41  VaR(N)=1.99%  VaR(H)=2.27%  Sharpe=2.50  Sortino=2.11
✅ ORCL: Vol=48.95%  Beta=0.76  VaR(N)=5.73%  VaR(H)=5.48%  Sharpe=-3.34  Sortino=-3.24
✅ ABBV: Vol=28.41%  Beta=0.35  VaR(N)=2.83%  VaR(H)=1.56%  Sharpe=1.03  Sortino=1.56
✅ SONY: Vol=30.80%  Beta=1.48  VaR(N)=3.33%  VaR(H)=2.30%  Sharpe=-1.16  Sortino=-1.18

Saved risk metrics  → /Users/ishanlahiru/Documents/invesment-portfoli

/Users/ishanlahiru/Documents/invesment-portfolio-analysis/backend/risk.py:174: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "as_of_utc": datetime.utcnow().isoformat(timespec="seconds"),


Saved AI risk summary → /Users/ishanlahiru/Documents/invesment-portfolio-analysis/processed/risk_report.json


CompletedProcess(args=['/Users/ishanlahiru/miniforge3/bin/python', '/Users/ishanlahiru/Documents/invesment-portfolio-analysis/frontend/risk_analyzer.py'], returncode=0)

In [384]:
import json
from pathlib import Path

# --- Load risk report JSON ---
risk_path = Path.home() / "Documents/invesment-portfolio-analysis/processed/risk_report.json"
risk_data = json.loads(risk_path.read_text(encoding="utf-8"))

# --- Prepare cards ---
cards = []
for idx, row in combined_df.iterrows():
    symbol = row["symbol"]

    # Get latest news for the stock
    news_list = get_stock_news(symbol, count=5)

    # Get risk info
    risk_info = risk_data.get(symbol, {})  # default empty dict if symbol missing

    # Prepare card
    card = {
        "symbol": symbol,
        "lastTradedTime": str(row["lastTradedTime"]),
        "current_price": row["current_price"],
        "pred_5d_price": row["pred_5d_price"],
        "pred_10d_price": row["pred_10d_price"],
        "news": news_list,
        "risk": risk_info  # add risk info here
    }
    cards.append(card)

# --- Convert to JSON string ---
cards_json = json.dumps(cards, indent=2)

# --- Save JSON to processed-rag folder ---
save_dir = Path.home() / "Documents/invesment-portfolio-analysis/processed-rag"
save_dir.mkdir(parents=True, exist_ok=True)
json_file_path = save_dir / "cards.json"

with open(json_file_path, "w", encoding="utf-8") as f:
    f.write(cards_json)

print(f"Cards JSON saved at: {json_file_path}")


Cards JSON saved at: /Users/ishanlahiru/Documents/invesment-portfolio-analysis/processed-rag/cards.json


In [382]:


prompt = """
You are a highly experienced financial analyst specializing in stock market evaluation and investment risk assessment.

Analyze the following stock data (in JSON format) and generate a structured summary suitable for **frontend display** and **human reading**.

Format each stock as follows in Markdown:
#### **<SYMBOL> (<Company Name>)**
- **Current Price:** $<current_price>  
- **Predicted 5-Day:** $<pred_5d> → indicate bullish/neutral/bearish trend  
- **Predicted 10-Day:** $<pred_10d> → indicate medium-term outlook  
- **Risk:** <RiskLevel> (Volatility <volatility>%, Beta <beta>)  
- **Investment Insight:** Short advice  
- **News Impact:** Summarize top headlines  
**Final Verdict:** <brief verdict with confidence level>

Return output in Markdown. Input JSON:
"""


cards_json = json.dumps(cards, indent=2)
final_prompt = prompt + cards_json


response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a professional financial analyst who outputs clean Markdown summaries and JSON for database use."},
        {"role": "user", "content": final_prompt}
    ],
    temperature=0.45,
    max_tokens=1800
)


markdown_output = response.choices[0].message.content
display(Markdown(markdown_output))


stocks_json_list = []

lines = markdown_output.split("\n")
current_stock = {}
for line in lines:
    line = line.strip()
    if line.startswith("#### **"):
        if current_stock:
            stocks_json_list.append(current_stock)
        current_stock = {"summary": line, "details": []}
    elif current_stock:
        if line:
            current_stock["details"].append(line)

if current_stock:
    stocks_json_list.append(current_stock)


save_dir = os.path.join(BASE_DIR, "processed-rag")
os.makedirs(save_dir, exist_ok=True)

json_path = os.path.join(save_dir, "financial_analysis.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(stocks_json_list, f, indent=2)

print(f"Financial analysis saved as JSON file:\n{json_path}")


```markdown
#### **TSM (Taiwan Semiconductor Manufacturing Company)**
- **Current Price:** $290.73  
- **Predicted 5-Day:** $294.18 → Bullish trend  
- **Predicted 10-Day:** $309.34 → Positive medium-term outlook  
- **Risk:** High (Volatility 47.5%, Beta 2.68)  
- **Investment Insight:** Not Best: High volatility and risk  
- **News Impact:** Intel's strong Q3 earnings boost chip demand; potential investment opportunities highlighted.  
**Final Verdict:** Cautious optimism; consider risk tolerance.

---

#### **ORCL (Oracle Corporation)**
- **Current Price:** $280.07  
- **Predicted 5-Day:** $281.14 → Neutral trend  
- **Predicted 10-Day:** $285.12 → Slightly positive medium-term outlook  
- **Risk:** High (Volatility 48.9%, Beta 0.76)  
- **Investment Insight:** Not Best: High volatility and risk  
- **News Impact:** Analyst interest in Oracle; speculation about future growth amidst high volatility.  
**Final Verdict:** High risk; monitor closely for better entry points.

---

#### **ABBV (AbbVie Inc.)**
- **Current Price:** $228.25  
- **Predicted 5-Day:** $225.17 → Bearish trend  
- **Predicted 10-Day:** $232.07 → Slightly positive medium-term outlook  
- **Risk:** Medium (Volatility 28.4%, Beta 0.35)  
- **Investment Insight:** Best for moderate-risk portfolios  
- **News Impact:** Positive analyst sentiment; navigating challenges in the pharma sector.  
**Final Verdict:** Good for conservative investors; stable outlook.

---

#### **SONY (Sony Group Corporation)**
- **Current Price:** $28.71  
- **Predicted 5-Day:** $28.49 → Bearish trend  
- **Predicted 10-Day:** $28.24 → Continued bearish outlook  
- **Risk:** High (Volatility 30.8%, Beta 1.48)  
- **Investment Insight:** Not Best: High volatility and risk  
- **News Impact:** Mixed sentiment; concerns over AI impact on streaming and gaming sectors.  
**Final Verdict:** High risk; consider alternatives.

---

#### **NKE (Nike, Inc.)**
- **Current Price:** $69.68  
- **Predicted 5-Day:** $69.18 → Neutral trend  
- **Predicted 10-Day:** $68.31 → Bearish outlook  
- **Risk:** High (Volatility 36.3%, Beta 1.71)  
- **Investment Insight:** Not Best: High volatility and risk  
- **News Impact:** Innovations and partnerships could drive future growth; however, current trends are concerning.  
**Final Verdict:** High risk; reassess investment strategy.

---

#### **CRM (Salesforce, Inc.)**
- **Current Price:** $255.05  
- **Predicted 5-Day:** $252.37 → Bearish trend  
- **Predicted 10-Day:** $249.88 → Continued bearish outlook  
- **Risk:** High (Volatility 37.5%, Beta 0.97)  
- **Investment Insight:** Not Best: High volatility and risk  
- **News Impact:** Market reactions to earnings and buybacks; cautious sentiment prevails.  
**Final Verdict:** High risk; maintain caution.

---

#### **SAP (SAP SE)**
- **Current Price:** $278.53  
- **Predicted 5-Day:** $275.68 → Bearish trend  
- **Predicted 10-Day:** $269.10 → Continued bearish outlook  
- **Risk:** Low (Volatility 21.3%, Beta 1.41)  
- **Investment Insight:** Best: Low volatility and risk  
- **News Impact:** Solid earnings amid challenges; potential for recovery noted.  
**Final Verdict:** Good for conservative investors; stable outlook.

---

#### **AAPL (Apple Inc.)**
- **Current Price:** $259.58  
- **Predicted 5-Day:** $263.39 → Bullish trend  
- **Predicted 10-Day:** $243.59 → Bearish medium-term outlook  
- **Risk:** Medium (Volatility 22.6%, Beta 1.25)  
- **Investment Insight:** Best for moderate-risk portfolios  
- **News Impact:** Strong earnings quality; concerns over legal challenges noted.  
**Final Verdict:** Balanced risk; potential for growth.

---

#### **TSLA (Tesla, Inc.)**
- **Current Price:** $448.98  
- **Predicted 5-Day:** $453.10 → Bullish trend  
- **Predicted 10-Day:** $410.51 → Bearish medium-term outlook  
- **Risk:** High (Volatility 50.6%, Beta 2.71)  
- **Investment Insight:** Not Best: High volatility and risk  
- **News Impact:** Mixed sentiment on earnings and market performance; investor concerns about margins.  
**Final Verdict:** High risk; monitor closely.

---

#### **INTC (Intel Corporation)**
- **Current Price:** $38.16  
- **Predicted 5-Day:** $37.75 → Bearish trend  
- **Predicted 10-Day:** $32.10 → Continued bearish outlook  
- **Risk:** High (Volatility 56.5%, Beta 1.26)  
- **Investment Insight:** Not Best: High volatility and risk  
- **News Impact:** Strong earnings report; concerns about market share recovery persist.  
**Final Verdict:** High risk; reassess investment strategy.
```

Financial analysis saved as JSON file:
/Users/ishanlahiru/Documents/invesment-portfolio-analysis/processed-rag/financial_analysis.json
